# Qwen2-VL LGT Order Refine Test Inference Only

Runs test inference only for this Qwen2 adapter:

`/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter`

No model comparison and no quick/tuning/holdout evaluation.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path('/content/.snu_qwen2_lgt_test_deps_installed')

if not MARKER.exists():
    packages = [
        'transformers>=4.49.0',
        'accelerate>=0.34.0',
        'bitsandbytes>=0.46.1',
        'peft',
        'qwen-vl-utils',
        'modelscope',
        'jedi',
        'pandas==2.2.2',
        'safetensors>=0.4.5',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])
    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime. Run this cell again after restart.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed. Continue.')

In [ ]:
# 2) Setup
from google.colab import drive
drive.mount('/content/drive')

import gc
import glob
import hashlib
import itertools
import json
import math
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
import random
import re
import shutil
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
from peft import PeftModel
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

ZIP_PATH = '/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip'
DATA_DIR = '/content/snuaichallenge_data'
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
TEST_IMAGE_DIR = os.path.join(DATA_DIR, 'test')

MODEL_REPO_ID = 'Qwen/Qwen2-VL-2B-Instruct'
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct'
ADAPTER_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter'

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = '/content/drive/MyDrive/SNU_AI_Challenge/qwen2_lgt_order_refine_test_only'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, RUN_ID)
CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache')
SUBMIT_PATH = os.path.join(OUTPUT_DIR, 'submission_qwen2_lgt_order_refine.csv')
for path in [OUTPUT_ROOT, OUTPUT_DIR, CACHE_DIR]:
    os.makedirs(path, exist_ok=True)

SEED = 42
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
STRUCTURED_ALPHA = 1.0
STRUCTURED_BETA = 1.0
STRUCTURED_GAMMA = 1.0
PAIRWISE_BIDIRECTIONAL = False
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall('/content/')

assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')), ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

test_df = pd.read_csv(TEST_CSV)
test_df['Id'] = test_df['Id'].astype(str)

print('adapter:', ADAPTER_DIR)
print('output:', OUTPUT_DIR)
print('test rows:', len(test_df))

In [ ]:
# 3) Config and IO helpers
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def append_jsonl(record, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def model_cache_is_complete(model_dir):
    if not os.path.exists(os.path.join(model_dir, 'config.json')):
        return False
    has_weight = any(os.path.exists(os.path.join(model_dir, name)) for name in ['model.safetensors.index.json', 'pytorch_model.bin', 'pytorch_model.bin.index.json']) or bool(glob.glob(os.path.join(model_dir, '*.safetensors')))
    has_processor = any(os.path.exists(os.path.join(model_dir, name)) for name in ['preprocessor_config.json', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json'])
    return bool(has_weight and has_processor)


def ensure_base_model_path():
    if model_cache_is_complete(DRIVE_MODEL_DIR):
        print('Using cached base model:', DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    if not USE_MODELSCOPE_BASE_MODEL:
        return MODEL_REPO_ID
    try:
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir='/content/modelscope_cache')
        os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
        if not model_cache_is_complete(DRIVE_MODEL_DIR):
            tmp = DRIVE_MODEL_DIR + '.tmp'
            if os.path.exists(tmp):
                shutil.rmtree(tmp)
            shutil.copytree(model_dir, tmp, dirs_exist_ok=True)
            if os.path.exists(DRIVE_MODEL_DIR):
                shutil.rmtree(DRIVE_MODEL_DIR)
            os.replace(tmp, DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    except Exception as exc:
        print('ModelScope download failed; using HF repo id:', repr(exc))
        return MODEL_REPO_ID


def nested_get(mapping, path, default=None):
    current = mapping if isinstance(mapping, dict) else {}
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def first_not_none(*values):
    for value in values:
        if value is not None:
            return value
    return None


def load_reference_config():
    global STRUCTURED_ALPHA, STRUCTURED_BETA, STRUCTURED_GAMMA, PAIRWISE_BIDIRECTIONAL, MIN_PIXELS, MAX_PIXELS
    candidates = [
        os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(ADAPTER_DIR))), 'run_config.json'),
        os.path.join(os.path.dirname(os.path.dirname(ADAPTER_DIR)), 'run_config.json'),
        os.path.join(os.path.dirname(ADAPTER_DIR), 'run_config.json'),
        os.path.join(ADAPTER_DIR, 'best_config.json'),
        os.path.join(ADAPTER_DIR, 'run_config.json'),
    ]
    merged = {}
    used = []
    for path in candidates:
        if os.path.exists(path):
            cfg = load_json(path)
            merged.update(cfg)
            used.append(path)
    candidate_cfg = first_not_none(merged.get('candidate_generator'), {})
    decoding_cfg = first_not_none(merged.get('decoding'), {})
    STRUCTURED_ALPHA = float(first_not_none(nested_get(candidate_cfg, ['alpha']), nested_get(decoding_cfg, ['alpha']), merged.get('alpha'), STRUCTURED_ALPHA))
    STRUCTURED_BETA = float(first_not_none(nested_get(candidate_cfg, ['beta']), nested_get(decoding_cfg, ['beta']), merged.get('beta'), STRUCTURED_BETA))
    STRUCTURED_GAMMA = float(first_not_none(nested_get(candidate_cfg, ['gamma']), nested_get(decoding_cfg, ['gamma']), merged.get('gamma'), STRUCTURED_GAMMA))
    PAIRWISE_BIDIRECTIONAL = bool(first_not_none(nested_get(candidate_cfg, ['pairwise_bidirectional']), merged.get('pairwise_bidirectional'), PAIRWISE_BIDIRECTIONAL))
    MIN_PIXELS = int(first_not_none(nested_get(candidate_cfg, ['min_pixels']), merged.get('min_pixels'), MIN_PIXELS))
    MAX_PIXELS = int(first_not_none(nested_get(candidate_cfg, ['max_pixels']), merged.get('max_pixels'), MAX_PIXELS))
    summary = {
        'used_config_paths': used,
        'alpha': STRUCTURED_ALPHA,
        'beta': STRUCTURED_BETA,
        'gamma': STRUCTURED_GAMMA,
        'pairwise_bidirectional': PAIRWISE_BIDIRECTIONAL,
        'min_pixels': MIN_PIXELS,
        'max_pixels': MAX_PIXELS,
    }
    save_json(summary, os.path.join(OUTPUT_DIR, 'reference_config_summary.json'))
    print(json.dumps(summary, ensure_ascii=False, indent=2))


load_reference_config()

In [ ]:
# 4) Model and prompt helpers
def load_model_class():
    if Qwen2VLForConditionalGeneration is not None:
        return Qwen2VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError('No compatible Qwen2-VL model class found.')


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def model_device(active_model):
    return next(active_model.parameters()).device


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f'{value!r} tokenized to {ids}; this notebook expects single-token labels.')
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}


def chat_prompt(messages, add_generation_prompt=True):
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)


def user_message_with_images(image_count, instruction):
    content = []
    for idx in range(1, image_count + 1):
        content.append({'type': 'text', 'text': f'\nImage {idx}:'})
        content.append({'type': 'image'})
    content.append({'type': 'text', 'text': '\n\n' + instruction})
    return [{'role': 'user', 'content': content}]


def row_image_paths(row, image_root):
    sample_id = str(row['Id'])
    return [os.path.join(image_root, sample_id, str(row[f'Input_{i}'])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert('RGB').copy()


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def task_instruction(example):
    sentence = example['sentence']
    task_type = example['task_type']
    if task_type == 'pairwise':
        return (
            f'Caption:\n{sentence}\n\n'
            'Question: Which image occurs first?\n'
            'If the first image occurs earlier, answer 1.\n'
            'If the second image occurs earlier, answer 2.\n'
            'Answer only 1 or 2.'
        )
    if task_type == 'first':
        return f'Caption:\n{sentence}\n\nQuestion: Which image represents the beginning of the story?\nAnswer only the image number from 1 to 4.'
    if task_type == 'last':
        return f'Caption:\n{sentence}\n\nQuestion: Which image represents the end of the story?\nAnswer only the image number from 1 to 4.'
    raise ValueError(task_type)


def make_example(row, task_type, image_root, pair=None):
    image_paths = row_image_paths(row, image_root)
    example = {
        'sample_id': str(row['Id']),
        'sentence': '' if pd.isna(row['Sentence']) else str(row['Sentence']),
        'image_paths': image_paths,
        'task_type': task_type,
        'target': '1',
    }
    if task_type == 'pairwise':
        a, b = pair
        example['image_paths'] = [image_paths[a - 1], image_paths[b - 1]]
    return example


def messages_for_example(example):
    return user_message_with_images(len(example['image_paths']), task_instruction(example))


def load_adapter_model():
    base = load_model_class().from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map='auto',
        local_files_only=MODEL_LOCAL_FILES_ONLY,
        trust_remote_code=True,
    )
    active_model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
    active_model.eval()
    active_model.config.use_cache = True
    if hasattr(active_model, 'generation_config'):
        active_model.generation_config.do_sample = False
        active_model.generation_config.temperature = None
        active_model.generation_config.top_p = None
        active_model.generation_config.top_k = None
        active_model.generation_config.num_beams = 1
    return active_model


model = load_adapter_model()
print('loaded Qwen2 adapter')

In [ ]:
# 5) Structured inference
@torch.inference_mode()
def score_digit_candidate_batch(active_model, examples, candidates_per_example):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = 'right'
        texts = [chat_prompt(messages_for_example(example), add_generation_prompt=True) for example in examples]
        image_cache = {}

        def cached_load(path):
            if path not in image_cache:
                image_cache[path] = load_rgb(path)
            return image_cache[path]

        images = [[cached_load(path) for path in example['image_paths']] for example in examples]
        inputs = processor(text=texts, images=images, padding=True, return_tensors='pt')
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        result = []
        for row_index, candidates in enumerate(candidates_per_example):
            last_pos = int(inputs['attention_mask'][row_index].sum().item()) - 1
            logits = outputs.logits[row_index, last_pos]
            token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
            probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
            result.append({int(candidate): float(prob) for candidate, prob in zip(candidates, probs)})
        return result
    finally:
        processor.tokenizer.padding_side = old_padding_side


def structured_score(first_probs, last_probs, pair_probs, order, alpha=STRUCTURED_ALPHA, beta=STRUCTURED_BETA, gamma=STRUCTURED_GAMMA):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(pair_probs[f'{order[i]}>{order[j]}']) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(first_probs[str(order[0])]) + eps)
    last_score = math.log(float(last_probs[str(order[-1])]) + eps)
    return float(alpha * pair_score + beta * first_score + gamma * last_score)


def infer_one(row):
    examples = [make_example(row, 'first', TEST_IMAGE_DIR), make_example(row, 'last', TEST_IMAGE_DIR)]
    candidates = [[1, 2, 3, 4], [1, 2, 3, 4]]
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        examples.append(make_example(row, 'pairwise', TEST_IMAGE_DIR, pair=(a, b)))
        candidates.append([1, 2])
        if PAIRWISE_BIDIRECTIONAL:
            examples.append(make_example(row, 'pairwise', TEST_IMAGE_DIR, pair=(b, a)))
            candidates.append([1, 2])

    probs = score_digit_candidate_batch(model, examples, candidates)
    first_probs = {str(k): v for k, v in probs[0].items()}
    last_probs = {str(k): v for k, v in probs[1].items()}
    pair_probs = {}
    offset = 2
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        forward = probs[offset]
        offset += 1
        p_a_before_b = float(forward[1])
        if PAIRWISE_BIDIRECTIONAL:
            reverse = probs[offset]
            offset += 1
            p_a_before_b = 0.5 * (p_a_before_b + float(reverse[2]))
        pair_probs[f'{a}>{b}'] = p_a_before_b
        pair_probs[f'{b}>{a}'] = 1.0 - p_a_before_b

    candidates24 = []
    for order in PERMUTATIONS:
        score = structured_score(first_probs, last_probs, pair_probs, order)
        candidates24.append({'order': list(order), 'structured_score': score})
    candidates24 = sorted(candidates24, key=lambda item: item['structured_score'], reverse=True)
    pred_order = candidates24[0]['order']
    cache = {
        'sample_id': str(row['Id']),
        'first_probs': first_probs,
        'last_probs': last_probs,
        'pair_probs': pair_probs,
        'candidate_orders': candidates24,
        'pred_order': pred_order,
    }
    return pred_order, cache

In [ ]:
# 6) Run test inference and save submission
submission_rows = []
cache_path = os.path.join(CACHE_DIR, 'test_structured_cache.jsonl')
if os.path.exists(cache_path):
    os.remove(cache_path)

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='qwen2 lgt test'):
    pred_order, cache = infer_one(row)
    submission_rows.append({'Id': str(row['Id']), 'Answer': str(sequence_to_answer(pred_order))})
    append_jsonl(cache, cache_path)

submission = pd.DataFrame(submission_rows)
submission.to_csv(SUBMIT_PATH, index=False)

run_config = {
    'adapter_dir': ADAPTER_DIR,
    'model_repo_id': MODEL_REPO_ID,
    'model_id': MODEL_ID,
    'alpha': STRUCTURED_ALPHA,
    'beta': STRUCTURED_BETA,
    'gamma': STRUCTURED_GAMMA,
    'pairwise_bidirectional': PAIRWISE_BIDIRECTIONAL,
    'min_pixels': MIN_PIXELS,
    'max_pixels': MAX_PIXELS,
    'test_rows': len(test_df),
    'submission_path': SUBMIT_PATH,
    'cache_path': cache_path,
}
save_json(run_config, os.path.join(OUTPUT_DIR, 'run_config.json'))

display(submission.head())
print('submission saved:', SUBMIT_PATH)
print('cache saved:', cache_path)